---
authors:
  - name: Giulia Cesaro
  - name: James Nagai
---


(diff-cell-cell)=
# Differential cell-cell communication

## Motivation

Cell–cell communication (CCC) is inherently context-dependent. Cellular interactions are dynamically shaped by biological factors such as genetic background, disease state, phenotype, developmental stage, treatment, and time. Consequently, the same cell types can exhibit distinct communication programs across different biological conditions and across individual patients.
As discussed on [Cell-Cell comunication](cell_cell_communication.ipynb) chapter, hundreds of computational methods have been developed to infer ligand–receptor (LR)-mediated communication from single-cell transcriptomic data {cite:p}`diff_ccc:armingol2024diversification,diff_ccc:cesaro2025advances`. Most of these methods primarly focus on inferring LR pairs within individual biological conditions. While these approaches have greatly advanced our understanding of tissue organization and signaling, they are not specifically designed to characterize identify how communication is rewired across conditions or to account for inter-patient variability.

The rapid adoption of single-cell RNA sequencing (scRNA-seq), together with the emergence of large-scale cell atlases and cohort studies, has fundamentally changed the landscape of CCC analysis. Modern datasets increasingly include dozens or hundreds of samples spanning multiple patients, disease states, treatments, developmental stages, or longitudinal time points. Rather than asking *which ligand–receptor interactions are active?*, researchers increasingly seek to determine *how communication changes between conditions, which signaling programs are consistently altered across patients, and how patient-to-patient heterogeneity influences cellular interactions*.

These developments have driven a shift from single-condition CCC inference toward comparative and differential communication analysis across multiple biological conditions or multiple samples. 


## Background

Comparative CCC methods build upon the same principles as single-condition CCC analysis. They typically infer ligand–receptor-mediated communication from single-cell RNA sequencing data by using gene expression as a proxy for protein abundance, requiring cells to be grouped into biologically meaningful clusters, and integrating prior biological knowledge through curated ligand–receptor interaction resources, transcriptional regulatory networks and/or signaling pathway databases. For a detailed overview of the principles, assumptions, and workflow underlying single-condition CCC inference, see Chapter [Cell-Cell comunication](cell_cell_communication.ipynb).
The key distinction lies in the computational framework used to compare inferred communication patterns across biological conditions or multiple patient samples. Rather than identifying communication events within a single dataset, these methods quantify and evaluate changes in ligand–receptor interactions, enabling the detection of signaling programs that are differentially regulated across biological contexts.

As highlighted by Cesaro et al. {cite:p}`diff_ccc:cesaro2025advances`, comparative CCC methods employ diverse computational strategies, ranging from set-based comparisons and statistical testing (e.g., differential expression of ligands and receptors and statistical testing of communication scores) to network-based analyses and machine learning techniques for identifying differential communication patterns. Beyond intercellular communication, some methods also investigate changes in downstream intracellular signaling by integrating prior knowledge of signaling pathways and gene regulatory networks to identify altered signaling cascades and transcriptional responses across biological conditions.

Most methods rely on pairwise condition-level comparisons, identifying differences in communication patterns between two biological conditions. More recently, sample-level approaches have emerged to characterize sample-specific communication patterns, capture inter-sample variability, and investigate communication heterogeneity across individuals. In this chapter, we present an example of condition-based differential cell–cell communication analysis, illustrating the comparison between two experimental conditions. We then extend this framework to sample-level CCC analysis, highlighting approaches that capture communication variability across multiple samples or patient cohorts.

Figure?

## Environment setup


In the first part of this chapter, we illustrate condition-level differential CCC analysis through a practical comparison of two experimental conditions. We first use pyCrossTalkeR ({cite:p}`diff_ccc:nagai2021crosstalker`), that integrates inferred ligand–receptor interactions into cell–cell communication networks and applies graph-based metrics to identify changes in intercellular signaling between biological conditions. We then introduce NicheNetpy, which integrates prior knowledge of ligand–receptor interactions, intracellular signaling, and gene regulatory networks to predict ligands potentially responsible for observed changes in gene expression in receiver cells between conditions, thereby linking intercellular communication to downstream transcriptional responses.

In the second part, we focus on sample-level CCC analysis, using scAccordion ({cite:p `diff_ccc:nagai2025scaccordion`)to infer and compare communication patterns across individual biological samples. 

In [1]:
import warnings

warnings.filterwarnings("ignore")

In [2]:
import lamindb as ln
import pandas as pd
import scanpy as sc

ln.connect("theislab/sc-best-practices")

ln.track("8YR6fz482TZ3")

sc.settings.set_figure_params(dpi=200, frameon=False)
sc.set_figure_params(dpi=200, facecolor="white")
sc.set_figure_params(figsize=(5, 5))

→ connected lamindb: theislab/sc-best-practices
! connected in read-only mode, please use ln.DB("theislab/sc-best-practices") instead
! skipping track(), connected in read-only mode


## Case study

We demonstrate comparative CCC analysis using human myocardial infarction single-nucleus RNA sequencing (snRNA-seq) data from the study *Spatial multi-omic map of human myocardial infarction* ({cite:p}`diff_ccc:kuppe2022spatial`). We focus on two conditions: myogenic (control) and ischemic (disease), aiming to identify changes in cell–cell communication associated with myocardial infarction.

We first download the preprocessed dataset for downstream analysis.

In [3]:
# JAMES
import liana as li
# load data as from CCC chapter
adata = li.ds.kang_2018()

In [4]:
sample_key = 'sample'
condition_key = 'condition'
groupby = 'cell_type'

Show pre-computed {term}`UMAP <Uniform manifold approximation and projection (UMAP)>`, just to showcase the data

### Condition-based Differential CCC Analysis

The aim is to identify cell-cell communication signals that are differentially regulated between the two conditions, thereby highlighting signaling interactions that are gained, lost, or altered between the conditions.


Before running pyCrossTalkeR, we first need to infer ligand–receptor interactions from our dataset. To this end, we use the LIANA framework as used in the [Cell-Cell comunication](cell_cell_communication.ipynb) chapter. Here, we use scSeqComm ({cite:p}`diff_ccc:baruzzo2022identify`) as an example to predict ligand–receptor interactions, which will subsequently serve as input for comparative CCC analysis with pyCrossTalkeR.

In [5]:
import liana as li

li.mt.scseqcomm.by_sample(
    adata,
    groupby=groupby,
    resource_name='consensus', # NOTE: uses human gene symbols!
    sample_key=condition_key, # sample key by which we which to loop
    use_raw=False,
    verbose=True, # use 'full' to show all verbose information
    n_perms=10, # exclude permutations for speed
    return_all_lrs=False# return all LR values
    )

Now running: stim: 100%|██████████| 2/2 [00:03<00:00,  1.99s/it]


Next, we use pyCrossTalkeR to perform graph-based differential cell–cell communication analysis, comparing the inferred communication networks between the two biological conditions to identify changes in signaling interactions and communication patterns.

In [6]:
from pycrosstalker import tools as cttl
from pycrosstalker import plots as ctpl

In [7]:
adata = cttl.utils.from_liana(adata,
                              liana_key = "liana_res",
                              compute_means=True,pval_filter=False)
adata = cttl.analise_LR(adata,save=False)

Create a Differential Table
Calculating CCI Ranking
Calculating GCI Ranking
Network Analysis Done
Generating h5ad file with Analysed Results


We use plot_cci to visualize differential communication between ischemic and myogenic conditions. Nodes represent cell types, with size proportional to the absolute change in PageRank centrality, highlighting changes in their network importance. Edges represent differential signaling, with red indicating stronger communication in ischemic and blue in myogenic, while edge width reflects the magnitude of the difference.

### Differential Intracellular Analysis

TO DO: NICHENETPY

##  Overview

to do

## Quiz

In [11]:
%run ../../scripts/quiz.py

with quiz_tabs():
    flip_card(
        "q1",
        "What is the main difference between condition-level and sample-level CCC analysis?",
        "Condition-level analysis compares CCC patterns between biological conditions, often pooling samples, while sample-level analysis infers CCC for each sample independently, capturing inter-sample variability and heterogeneity.",
    )
